* Make sure to first run the ingest pipeline to simulate already existing data

In [ ]:
import sys

from pyspark.sql import SparkSession

JOB_PATH = '/opt/spark/jobs'
if JOB_PATH not in sys.path:
    sys.path.insert(0, JOB_PATH)

from helpers import *
from metadata import *
from silver_job import silver_job
from bronze_job import bronze_job

* Run an update pipeline with no validations (all input config files should have empty arrays in the `quality_checks` and `cross_batch_quality_checks` fields)

* (`taxi_trips_v2.json`, `air_quality_v2.json`,  `weather_v2.json`)

In [ ]:
spark = (
    SparkSession.builder
    .appName(f"ValidationOverheadOfUpdate")
    .config("spark.executor.memory", "2g")
    .config("spark.sql.maxPlanStringLength", "8192")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .getOrCreate()
)
spark.conf.set("spark.sql.session.timeZone", "UTC")
spark.sparkContext.setLogLevel("ERROR")

run_context = create_run_context()

config_paths = [
    "/opt/spark/configs/taxi_trips_v2.json",
    "/opt/spark/configs/air_quality_v2.json",
    "/opt/spark/configs/weather_v2.json"
]

for config_path in config_paths:
    config = load_config(config_path)
    dataset = config["dataset"]
    print(f"Loading dataset: {dataset}")
    bronze_job(spark, config, run_context, mode="append")

for config_path in config_paths:
    config = load_config(config_path)

    silver_job(spark, config, run_context, incremental=True)


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/27 17:30:29 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Loading dataset: taxi_trips
bronze_job
read_landing_and_add_metadata


Source files:
  file:///opt/spark/landing/taxi/yellow_tripdata_2025-01_ext.parquet


write_bronze


SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.


Written bronze Delta table to: s3a://delta-data/bronze/bronze_taxi_trips
record_execution


... (6 outputs left out)

Incremental silver job for air_quality: filtering bronze data ingested at >= 2026-09-27 17:18:18.954460


Written silver Delta table to: s3a://delta-data/silver/silver_air_quality
record_execution


Incremental silver job for weather: filtering bronze data ingested at >= 2026-09-27 17:20:54.097369


Written silver Delta table to: s3a://delta-data/silver/silver_weather
record_execution


* See time results

In [ ]:
df = spark.read.format("delta").load("s3a://delta-data/metadata/pipeline_executions")

df.createOrReplaceTempView("pipeline_executions")

spark.sql(f"SELECT * FROM pipeline_executions WHERE pipeline_run_id = '{run_context['run_id']}'").show()

+--------------------+--------------------+------+-----------+--------------+--------------------+--------------------+----------------+-----------------+----------------+----------------+------------------------+
|     pipeline_run_id|        execution_id| stage|    dataset|schema_version|          started_at|         finished_at|duration_seconds|processed_records|inserted_records|rejected_records|validation_failure_count|
+--------------------+--------------------+------+-----------+--------------+--------------------+--------------------+----------------+-----------------+----------------+----------------+------------------------+
|d20027e8-cbe2-4d1...|f39ac66b-4b9e-427...|silver|air_quality|           2.0|2026-09-27 17:31:...|2026-09-27 17:32:...|           24.92|           584516|          584516|               0|                       0|
|d20027e8-cbe2-4d1...|1e6d2ba9-29f3-478...|bronze|air_quality|           2.0|2026-09-27 17:31:...|2026-09-27 17:31:...|           13.38|        

* Compare results with the `incremental_update_time` notebook times, that has validations enabled

In [ ]:
spark.stop()